# Person 2 — RQ3 and RQ4: Transportation research analysis

This analysis implements the team's assigned research questions using the checked-in integrated snapshot. It was added during integration repair; the recovered raw-source cleaning/EDA pipeline is documented separately in `Transport_Data_Cleaning.ipynb`.

Sources: [Housing hg8x-zxpr](https://data.cityofnewyork.us/d/hg8x-zxpr) and [SIRTA xeqp-qz8h](https://data.cityofnewyork.us/d/xeqp-qz8h).
Run from the repository root. No live download is required.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
sys.path.insert(0, str(Path("app").resolve()))
from charts.rq3_rq4 import rq3, rq4

df_integrated = pd.read_csv("df_integrated.csv", dtype={"zip_code": "string"})
df_transport_clean = pd.read_csv("df_transport_clean.csv")
assert not df_integrated.duplicated(["zip_code", "year"]).any()
assert not df_integrated.isna().any().any()
print("Integrated shape:", df_integrated.shape)
print("Cleaned transportation snapshot dates:", df_transport_clean.crash_date.min(), df_transport_clean.crash_date.max())
display(df_integrated.groupby("year")[["affordable_units", "serious_crash_count", "severe_injuries"]].sum())


Integrated shape: (484, 11)
Cleaned transportation snapshot dates: 2022-10-01 2026-06-30


,affordable_units,serious_crash_count,severe_injuries
year,,,
2022,15849.0,339.0,338.0
2023,28832.0,1553.0,1549.0
2024,28730.0,1597.0,1601.0
2025,28059.0,1846.0,1886.0
2026,11218.0,546.0,566.0


## RQ3 — How have affordable housing production and serious crashes changed over time?
Annual totals use two labelled axes because units and crash counts have different scales. Compare 2023–2025 for complete calendar-year windows (with missing-ZIP exclusions); 2022 and 2026 cover partial years. Matched ZIP coverage also changes by year, so this is not a citywide trend or a controlled comparison.


In [2]:
rq3(df_integrated).show()
annual = df_integrated.groupby("year")[["affordable_units", "serious_crash_count"]].sum()
full_years = annual.loc[2023:2025]
display(full_years)
display(full_years.pct_change().mul(100).rename(columns=lambda c: c + "_change_percent"))


,affordable_units,serious_crash_count
year,,
2023,28832.0,1553.0
2024,28730.0,1597.0
2025,28059.0,1846.0


,affordable_units_change_percent,serious_crash_count_change_percent
year,,
2023,NaN,NaN
2024,-0.353774,2.833226
2025,-2.335538,15.591735


Interpret the displayed totals and percentage changes together: the series can rise or fall at different rates. Two lines moving together do not demonstrate that housing causes crashes. These totals are restricted to ZIP-years represented in both datasets.

## RQ4 — Do ZIP codes with more housing projects also have more severe crash injuries?
One point per ZIP, summed across selected years. Report Pearson and rank correlations as descriptive associations. Annual project counts can count a project in more than one ZIP/year; they are not unique citywide projects. Repeat on 2023–2025 to inspect sensitivity to partial years.


In [3]:
rq4(df_integrated).show()
for label, data in [("All available years", df_integrated),
                    ("2023–2025", df_integrated[df_integrated.year.between(2023, 2025)])]:
    totals = data.groupby("zip_code")[["housing_project_count", "severe_injuries"]].sum()
    pearson = totals.housing_project_count.corr(totals.severe_injuries)
    spearman = totals.housing_project_count.rank().corr(totals.severe_injuries.rank())
    print(f"{label}: {len(totals)} ZIPs; Pearson r={pearson:.3f}; Spearman rho={spearman:.3f}")
    display(totals.nlargest(5, "severe_injuries"))


All available years: 143 ZIPs; Pearson r=0.712; Spearman rho=0.814


,housing_project_count,severe_injuries
zip_code,,
11207,50.0,125.0
11203,39.0,123.0
11234,4.0,119.0
11212,53.0,115.0
10456,63.0,112.0


2023–2025: 134 ZIPs; Pearson r=0.646; Spearman rho=0.759


,housing_project_count,severe_injuries
zip_code,,
11234,4.0,119.0
11207,36.0,101.0
10456,46.0,94.0
11212,39.0,94.0
11203,23.0,92.0


## Limits and source recovery
Population, road length, traffic volume and reporting coverage can influence both measures. Housing includes preservation. This notebook does not establish causal effects or per-person crash risk.

`Transport_Data_Cleaning.ipynb` now reproduces all supplied transportation fields from the archived official source. The historical source notebook is still absent, so the recovered method is attributed to the audit, rather than presented as evidence of original individual work. Dropping unknown ZIPs removes 5,682 raw records and explains the cleaned snapshot's later 2022 start. Source ZIP 10000 is flagged; it has no housing match.
